# htmx v4 Migration Tests

Automated browser tests for FastHTML apps migrated to htmx v4, using [solvecdp](https://github.com/AnswerDotAI/solvecdp) (a Python CDP client built on [solveit-chrome](https://github.com/AnswerDotAI/solveit-chrome)).

- `BASE_URL` must match your Solveit instance URL

In [ ]:
import asyncio
from solvecdp import JsCDP
import json

BASE_URL = 'https://htmx-new-v4.solveit.pub'

In [ ]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def run_dialog(name, sleep=0.2):
    await load_dialog(src_dname=name)
    await asyncio.sleep(sleep)
    try:
        yield
    finally:
        srv.stop()

## Test: echo4

In [ ]:
async with run_dialog("/fasthtml-example/echo/echo4"):
    page = await JsCDP.new(url=f'{BASE_URL}/?q=hello')
    result = await page.eval('document.body.innerText')
    assert result == 'hello', f"Expected 'hello', got '{result}'"
    print("✅ echo4 passed")
    await page.page.close()

✅ echo4 passed


## Test: ws_core4

In [ ]:
async with run_dialog("/fasthtml-example/htmx/ws_core4"):
    page = await JsCDP.new(url=BASE_URL)
    assert (await page.eval('notifications.innerText')) == "Hello, you have connected"
    root = await page.ax_tree()
    input_id = root.find_id('textbox')
    await page.fill_text(input_id, "Hoa")
    await page.eval('document.getElementById("form").requestSubmit(), true')
    await page.wait_for('notifications.innerText === "Hello Hoa"', timeout=3)
    await page.wait_for('notifications.innerText === "Goodbye Hoa"', timeout=3)
    await page.page.close()
    print("✅ ws_core4 passed")

✅ ws_core4 passed


## Test: sse_core4

In [ ]:
async with run_dialog("/fasthtml-example/htmx/sse_core4", sleep=0.5):
    page = await JsCDP.new(url=BASE_URL)
    await page.wait_for('document.querySelectorAll("article").length >= 2')
    print(f'✅ sse_core4: got streaming articles')
    await page.page.close()

✅ sse_core4: got streaming articles


## Test: oob4

In [ ]:
async with run_dialog("/fasthtml-example/htmx/oob4"):
    page = await JsCDP.new(url=f'{BASE_URL}')

    async def extract_content(column):
        content = await page.eval(f"document.getElementById('{column}').parentElement.textContent")
        return [s.strip() for s in content.split('\n') if s.strip()]

    # Click 1
    await page.eval("document.querySelector('button').click(); true")
    await page.wait_for_ready()

    first_text = await extract_content("first")
    second_text = await extract_content("second")
    third_text = await extract_content("third")

    assert first_text == ['first', 'thing E', 'thing D', 'thing C', 'thing A', 'thing B'], f"First click, first: {first_text}"
    assert second_text == ['second', 'thing F', 'thing G', 'thing H', 'thing I'], f"First click, second: {second_text}"
    assert third_text == ['third', 'thing J'], f"First click, third: {third_text}"

    # Click 2
    await page.eval("document.querySelector('button').click(); true")
    await page.wait_for_ready()

    first_text = await extract_content("first")
    second_text = await extract_content("second")
    third_text = await extract_content("third")

    assert first_text == ['first', 'thing E', 'thing D', 'thing C', 'thing A', 'thing B', 'thing E', 'thing D', 'thing C', 'thing A', 'thing B'], f"Second click, first: {first_text}"
    assert second_text == ['second', 'thing F', 'thing G', 'thing H', 'thing I'], f"Second click, second: {second_text}"
    assert third_text == ['third', 'thing J'], f"Second click, third: {third_text}"

    await page.page.close()
    print("✅ oob4 test passed!")

✅ oob4 test passed!


## Test: svgswap4

In [ ]:
async with run_dialog("/fasthtml-example/htmx/svgswap4"):
    page = await JsCDP.new(url=f'{BASE_URL}')

    # Initial state: blue circle exists
    assert await page.eval("document.getElementById('cts') !== null"), "Initial circle should exist"

    # Click "Multi" button → should show red and green circles
    await page.eval("document.getElementById('val').click(); true")
    await page.wait_for_ready()
    assert await page.eval("document.querySelector('circle[fill=red]') !== null"), "Red circle should appear"
    assert await page.eval("document.querySelector('circle[fill=green]') !== null"), "Green circle should appear"
    assert await page.eval("document.getElementById('val').textContent") == "Blue", "Button should say 'Blue'"

    # Click "Blue" button → should go back to blue circle
    await page.eval("document.getElementById('val').click(); true")
    await page.wait_for_ready()
    assert await page.eval("document.querySelector('circle[fill=blue]') !== null"), "Blue circle should reappear"
    assert await page.eval("document.getElementById('val').textContent") == "Multi", "Button should say 'Multi'"

    # Click blue circle → should swap to blue rectangle
    await page.eval("document.getElementById('cts').dispatchEvent(new MouseEvent('click', {bubbles: true}))")
    await page.wait_for_ready()
    assert await page.eval("document.querySelector('rect[fill=blue]') !== null"), "Blue rect should appear"

    await page.page.close()
    print("✅ svgswap4 passed")

✅ svgswap4 passed


## Test: counter-rsjs4

In [ ]:




















async with run_dialog("/fasthtml-example/htmx/counter-rsjs4"):
    async def get_outputs():
        return json.loads(await page.eval("JSON.stringify(Array.from(document.querySelectorAll('#my-output')).map(el => el.value))"))

    page = await JsCDP.new(url=BASE_URL)
    
    # Check initial values
    initial = await get_outputs()
    assert initial == ['0', '5'], f"Expected ['0', '5'], got {initial}"
    
    # Click both buttons
    await page.eval("document.querySelectorAll('[data-incrementer-increment]').forEach(btn => btn.click()); true")
    
    # Check final values
    final = await get_outputs()
    assert final == ['1', '6'], f"Expected ['1', '6'], got {final}"
    
    await page.page.close()
    print("✅ counter-rsjs4 passed")

✅ counter-rsjs4 passed


## Test: formdemo4

In [ ]:
async with run_dialog("/fasthtml-example/htmx/formdemo4"):
    page = await JsCDP.new(url=BASE_URL)

    async def result(): return await page.eval("document.getElementById('result').textContent")
    async def set_select(val): return await page.eval(f"(()=>{{ const s = document.getElementById('select'); s.value = '{val}'; s.dispatchEvent(new Event('input', {{bubbles: true}})); return true }})()")
    async def set_input(id, val):
        root = await page.ax_tree()
        await page.eval(f"(()=>{{ document.getElementById('{id}').value = ''; return true }})()")
        await page.fill_text(root.find_id('textbox', id.title()), val)

    await page.eval("document.querySelector('button').click(); true")
    await page.wait_for_ready()
    assert await result() == '{"a":"1","b":"2"}'

    await set_select('Two')
    await page.wait_for(f"document.getElementById('result').textContent === '{{\"select\":\"Two\",\"name\":\"j\",\"email\":\"h\"}}'")
    assert await result() == '{"select":"Two","name":"j","email":"h"}'

    await set_input('name', 'Hoa')
    await page.wait_for(f"document.getElementById('result').textContent.includes('Hoa')")
    assert await result() == '{"select":"Two","name":"Hoa","email":"h"}'

    await page.page.close()
    print("✅ formdemo4 passed")

✅ formdemo4 passed
